In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import os

# Publication style
plt.rcdefaults()
rcParams['font.family'] = 'Arial'
rcParams['font.size'] = 8
rcParams['axes.linewidth'] = 0.5
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
rcParams['xtick.major.width'] = 0.5
rcParams['ytick.major.width'] = 0.5
rcParams['xtick.major.size'] = 3
rcParams['ytick.major.size'] = 3
rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42

# Normalization, Spatial Binning and Data Export - Metabolic Gradient

This notebook normalizes fluorescence data against background, bins cells into spatial regions along the chamber axis, and exports the data files used by the figure notebooks.

**Input:** `analysis_code/0_combined_df_metabolic_gradient.csv` (from `0_combined_df_metabolic_gradient.ipynb`)

**Outputs:**
- `analysis_code/1_fluo_binary_metabolic_gradient.csv` — cell-level data (used by S7B and S7C)
- `S7A/1_summary_metabolic_gradient.csv` — per-position/bin/time aggregates (used by S7A)

In [2]:
# File paths and parameters
file_path = '0_combined_df_metabolic_gradient.csv'
file_path_background = '../../Figure2/2AB/analysis_code/background/0_combined_df_bg.csv'
replicate = 'replicate_1'

# Load data
df = pd.read_csv(file_path)
df = df[df['replicate'] == replicate]
df_bg = pd.read_csv(file_path_background)

# Normalize fluorescence: (intensity - background) / (frame_0_mean - background)
bg_intensity = df_bg[df_bg['frame_number'].isin(range(0, 9))]['intensity_raw_mcherry'].mean()
mean_intensity_frame_0 = df[df['frame_number'] == 0]['intensity_raw_mcherry'].mean()
df['fluo_norm'] = (df['intensity_raw_mcherry'] - bg_intensity) / (mean_intensity_frame_0 - bg_intensity)

print(f"Loaded {len(df)} rows for {replicate}")

Loaded 5505150 rows for replicate_1


In [3]:
# Bin chamber into spatial regions
bin_labels = {0: 'Back', 1: 'Middle', 2: 'Opening'}
df['y_bins'] = pd.cut(df['y'], bins=3, labels=False).astype(int)
df['y_bin_label'] = df['y_bins'].map(bin_labels)

# Convert frame number to hours (1 frame = 5 min)
df['time_hours'] = df['frame_number'] * (5 / 60)

# Save trimmed cell-level data (only columns needed downstream; gitignored)
cols = ['pos', 'y_bin_label', 'time_hours', 'intensity_raw_mcherry', 'fluo_norm']
df[cols].to_csv('1_fluo_binary_metabolic_gradient.csv', index=False)

# Save lightweight summary per position/bin/time for figure notebooks that
# only need aggregate statistics (no regression on individual cells)
summary = df.groupby(['pos', 'y_bin_label', 'time_hours']).agg(
    mean_intensity=('intensity_raw_mcherry', 'mean'),
    std_intensity=('intensity_raw_mcherry', 'std'),
    n_cells=('intensity_raw_mcherry', 'count'),
    y_min=('y', 'min'),
    y_max=('y', 'max'),
).reset_index()
summary.to_csv('../S7A/1_summary_metabolic_gradient.csv', index=False)
print(f"✅ Saved cell data ({len(cols)} cols) and summary ({len(summary)} rows)")

✅ Saved cell data (5 cols) and summary (4410 rows)
